# 1. Extracción y Caracterización de `seismic_data.csv`

**Trabajo Final - Grupo 06.** Evaluación del Desempeño Sísmico y Predicción del Índice de Daño Estructural con Machine Learning Supervisado.

**Fuente del dataset:** *Pre-Earthquake Prediction Dataset*.

Este cuaderno **extrae y carga** el dataset, verifica su integridad y caracteriza sus variables antes de construir el modelo predictivo.

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

CSV_NAME = "seismic_data.csv"
df = pd.read_csv(CSV_NAME)
df.head()

,Seismic Zone,PGA (m/s²),PGV (m/s),PGD (m),Spectral Acceleration (g),Soil Type,Site Amplification Factor,Historical Earthquake Magnitude (Mw),Fault Distance (km),Seismic Wave Frequency (Hz),Building Height (m),Number of Stories,Structural Material,Foundation Type,Natural Frequency (Hz),Damping Ratio (%),Mass of Structure (kg),Axial Stiffness (kN/m),Bending Stiffness (kN·m²),Lateral Load Resisting System,Predicted Max Inter-Story Drift Ratio (%),Predicted Max Roof Displacement (m),Predicted Base Shear Force (kN),Predicted Structural Acceleration (m/s²),Predicted Damage Index (0–1 Scale),Predicted Collapse Probability (%)
0,High,0.3789,0.3120,0.0572,0.6341,Sand,0.9392,6.8752,22.2034,1.6831,40.3941,17,Masonry,Raft,2.0003,3.0494,"737,892.3806","3,150.1717","25,771.7510",Moment Frame,2.1261,0.8398,"3,927.9404",0.7675,0.8045,80.4542
1,Very High,1.3463,1.2558,0.2045,1.7286,Clay,1.0898,6.8293,6.9472,4.0417,7.7264,48,Composite,Deep,2.4176,1.8901,"68,704.8464","3,245.3376","32,856.9450",Moment Frame,2.8803,1.5309,"2,923.7191",1.8482,0.7064,70.6411
2,Low,0.5024,0.6340,0.4433,0.1568,Rock,2.6892,7.1004,33.2248,5.8417,155.5021,34,Steel,Deep,0.2595,4.9007,"705,858.6930","7,518.1063","16,777.1713",Moment Frame,2.7629,0.7960,180.4469,4.4560,0.7956,79.5635
3,High,0.6149,0.4296,0.0246,1.0856,Rock,2.1943,7.8719,15.2251,9.4857,175.6317,14,Composite,Raft,2.1171,2.1350,"937,792.3075",781.1392,"3,013.8665",Braced Frame,2.9656,1.3039,"4,136.4264",3.7879,0.7668,76.6775
4,High,0.1813,0.8215,0.4807,0.3831,Rock,1.3570,8.8411,71.2834,0.7806,124.7617,17,Composite,Shallow,2.4100,4.9357,"728,244.9758","4,118.9847","25,287.5547",Braced Frame,0.8194,1.5196,926.8679,1.4601,0.2054,20.5368


## 2. Estructura del dataset

El archivo consta de 1000 registros numéricos y 26 variables, tal como se declara en el README del proyecto.

In [2]:
shape = pd.Series({"filas": df.shape[0], "columnas": df.shape[1]})
print(f"Dataset: {df.shape[0]} registros x {df.shape[1]} variables\n")
print(shape.to_string())

Dataset: 1000 registros x 26 variables

filas       1000
columnas      26


In [3]:
NUM = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
CAT = [c for c in df.columns if c not in NUM]

info = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "nulos": df.isna().sum(),
    "unicos": df.nunique(),
    "clase": ["numerica" if c in NUM else "categorica" for c in df.columns],
}).sort_values("clase")

print(f"Variables numericas: {len(NUM)}")
print(f"Variables categoricas: {len(CAT)}\n")
info

Variables numericas: 21
Variables categoricas: 5



,tipo,nulos,unicos,clase
Seismic Zone,str,0,4,categorica
Lateral Load Resisting System,str,0,3,categorica
Foundation Type,str,0,4,categorica
Soil Type,str,0,4,categorica
Structural Material,str,0,4,categorica
PGD (m),float64,0,1000,numerica
Predicted Structural Acceleration (m/s²),float64,0,1000,numerica
Predicted Base Shear Force (kN),float64,0,1000,numerica
Predicted Max Roof Displacement (m),float64,0,1000,numerica
Predicted Max Inter-Story Drift Ratio (%),float64,0,1000,numerica


## 3. Muestra de registros

In [4]:
df.head(5)

,Seismic Zone,PGA (m/s²),PGV (m/s),PGD (m),Spectral Acceleration (g),Soil Type,Site Amplification Factor,Historical Earthquake Magnitude (Mw),Fault Distance (km),Seismic Wave Frequency (Hz),Building Height (m),Number of Stories,Structural Material,Foundation Type,Natural Frequency (Hz),Damping Ratio (%),Mass of Structure (kg),Axial Stiffness (kN/m),Bending Stiffness (kN·m²),Lateral Load Resisting System,Predicted Max Inter-Story Drift Ratio (%),Predicted Max Roof Displacement (m),Predicted Base Shear Force (kN),Predicted Structural Acceleration (m/s²),Predicted Damage Index (0–1 Scale),Predicted Collapse Probability (%)
0,High,0.3789,0.3120,0.0572,0.6341,Sand,0.9392,6.8752,22.2034,1.6831,40.3941,17,Masonry,Raft,2.0003,3.0494,"737,892.3806","3,150.1717","25,771.7510",Moment Frame,2.1261,0.8398,"3,927.9404",0.7675,0.8045,80.4542
1,Very High,1.3463,1.2558,0.2045,1.7286,Clay,1.0898,6.8293,6.9472,4.0417,7.7264,48,Composite,Deep,2.4176,1.8901,"68,704.8464","3,245.3376","32,856.9450",Moment Frame,2.8803,1.5309,"2,923.7191",1.8482,0.7064,70.6411
2,Low,0.5024,0.6340,0.4433,0.1568,Rock,2.6892,7.1004,33.2248,5.8417,155.5021,34,Steel,Deep,0.2595,4.9007,"705,858.6930","7,518.1063","16,777.1713",Moment Frame,2.7629,0.7960,180.4469,4.4560,0.7956,79.5635
3,High,0.6149,0.4296,0.0246,1.0856,Rock,2.1943,7.8719,15.2251,9.4857,175.6317,14,Composite,Raft,2.1171,2.1350,"937,792.3075",781.1392,"3,013.8665",Braced Frame,2.9656,1.3039,"4,136.4264",3.7879,0.7668,76.6775
4,High,0.1813,0.8215,0.4807,0.3831,Rock,1.3570,8.8411,71.2834,0.7806,124.7617,17,Composite,Shallow,2.4100,4.9357,"728,244.9758","4,118.9847","25,287.5547",Braced Frame,0.8194,1.5196,926.8679,1.4601,0.2054,20.5368


## 4. Distribucion de las variables categoricas

In [5]:
for c in CAT:
    print(f"\n-- {c} ({df[c].nunique()} niveles) --")
    print(df[c].value_counts().to_string())


-- Seismic Zone (4 niveles) --
Seismic Zone
Very High    280
Low          258
High         232
Moderate     230

-- Soil Type (4 niveles) --
Soil Type
Rock         267
Sand         263
Clay         259
Soft Soil    211

-- Structural Material (4 niveles) --
Structural Material
Concrete     261
Steel        258
Composite    241
Masonry      240

-- Foundation Type (4 niveles) --
Foundation Type
Raft       266
Deep       258
Shallow    250
Pile       226

-- Lateral Load Resisting System (3 niveles) --
Lateral Load Resisting System
Shear Wall      350
Moment Frame    340
Braced Frame    310


## 5. Estadisticos descriptivos de las variables numericas

In [6]:
df[NUM].describe().T

,count,mean,std,min,25%,50%,75%,max
PGA (m/s²),"1,000.0000",0.7936,0.4048,0.1000,0.4403,0.7957,1.1477,1.4994
PGV (m/s),"1,000.0000",1.0412,0.5423,0.1001,0.5771,1.0508,1.4950,1.9957
PGD (m),"1,000.0000",0.2468,0.1443,0.0011,0.1249,0.2381,0.3712,0.4994
Spectral Acceleration (g),"1,000.0000",0.9449,0.4931,0.1067,0.5139,0.9410,1.3840,1.7991
Site Amplification Factor,"1,000.0000",1.8781,0.6276,0.8005,1.3317,1.8627,2.4045,2.9983
Historical Earthquake Magnitude (Mw),"1,000.0000",6.6788,1.2917,4.5011,5.5542,6.5909,7.7756,8.9851
Fault Distance (km),"1,000.0000",49.4806,27.6097,1.0052,26.9817,49.7277,72.5162,99.8631
Seismic Wave Frequency (Hz),"1,000.0000",5.1237,2.8544,0.1024,2.6135,5.1401,7.6230,9.9951
Building Height (m),"1,000.0000",103.8534,56.0508,5.4786,57.1655,103.7288,153.3358,199.9363
Number of Stories,"1,000.0000",24.2600,14.2916,1.0000,12.0000,24.0000,36.0000,49.0000


## 6. Verificacion de integridad

Comprobacion de duplicados, celdas vacias, columnas constantes y outliers extremos.

In [7]:
print(f"Registros duplicados exactos: {int(df.duplicated().sum())}")
print(f"Celdas vacias (NaN): {int(df.isna().sum().sum())}")
constantes = [c for c in df.columns if df[c].nunique() == 1]
print(f"Columnas constantes: {constantes if constantes else 'ninguna'}")

z = (df[NUM] - df[NUM].mean()) / df[NUM].std()
outliers = (z.abs() > 3).sum()
print(f"\nOutliers (|z| > 3): {int(outliers.sum())}")
if outliers.sum():
    print(outliers[outliers > 0].to_string())
else:
    print("Ninguna variable presenta outliers extremos.")

Registros duplicados exactos: 0
Celdas vacias (NaN): 0
Columnas constantes: ninguna

Outliers (|z| > 3): 0
Ninguna variable presenta outliers extremos.


## 7. Caracterizacion de la distribucion

La verificacion estadistica revela la naturaleza del dataset: todas las variables numéricas siguen una
distribución **uniforme** y las variables categóricas están equiprobables.

In [8]:
u = (df[NUM].max() - df[NUM].min()) / (4 * df[NUM].std())
print("Indicador de uniformidad  (max - min) / (4 * std)  ->  1.0 si la variable es uniforme")
print(u.round(3).to_string())

Indicador de uniformidad  (max - min) / (4 * std)  ->  1.0 si la variable es uniforme
PGA (m/s²)                                  0.8640
PGV (m/s)                                   0.8740
PGD (m)                                     0.8640
Spectral Acceleration (g)                   0.8580
Site Amplification Factor                   0.8750
Historical Earthquake Magnitude (Mw)        0.8680
Fault Distance (km)                         0.8950
Seismic Wave Frequency (Hz)                 0.8660
Building Height (m)                         0.8670
Number of Stories                           0.8400
Natural Frequency (Hz)                      0.8810
Damping Ratio (%)                           0.8680
Mass of Structure (kg)                      0.8620
Axial Stiffness (kN/m)                      0.8770
Bending Stiffness (kN·m²)                   0.8800
Predicted Max Inter-Story Drift Ratio (%)   0.8560
Predicted Max Roof Displacement (m)         0.8650
Predicted Base Shear Force (kN)             0.8

### 7.1 Identidad entre indice de dano y probabilidad de colapso

In [9]:
DAMAGE = next(c for c in df.columns if c.startswith("Predicted Damage Index"))
COLLAPSE = next(c for c in df.columns if c.startswith("Predicted Collapse Probability"))

r = df[DAMAGE].corr(df[COLLAPSE])
dif = np.abs(df[DAMAGE] * 100 - df[COLLAPSE]).max()
print(f"Pearson  r({DAMAGE}, {COLLAPSE}) = {r:.6f}")
print(f"Diferencia maxima |Dano x 100 - Colapso| = {dif:.2e}")
print("\nConclusion: son la MISMA variable expresada en escalas distintas (0-1 y 0-100 %).")

Pearson  r(Predicted Damage Index (0–1 Scale), Predicted Collapse Probability (%)) = 1.000000
Diferencia maxima |Dano x 100 - Colapso| = 1.42e-14

Conclusion: son la MISMA variable expresada en escalas distintas (0-1 y 0-100 %).


### 7.2 Correlacion entre las variables de entrada

In [17]:
INPUTS = [c for c in NUM if not c.startswith("Predicted")]
c = df[INPUTS].corr().abs()
iu = np.triu_indices(len(INPUTS), k=1)
pares = sorted(zip(c.to_numpy()[iu], iu[0], iu[1]), reverse=True)[:5]
top = pd.Series([round(v, 4) for v, _, _ in pares],
                index=[f"{INPUTS[i]} ~ {INPUTS[j]}" for _, i, j in pares])
print("Correlaciones mas altas entre variables de ENTRADA (Pearson |r|):")
print(top.to_string())
print("\nUn |r| maximo de 0.08 sobre 1000 muestras indica ausencia total de relacion.")

Correlaciones mas altas entre variables de ENTRADA (Pearson |r|):
PGV (m/s) ~ PGD (m)                               0.0794
Natural Frequency (Hz) ~ Mass of Structure (kg)   0.0737
PGV (m/s) ~ Natural Frequency (Hz)                0.0645
Building Height (m) ~ Axial Stiffness (kN/m)      0.0609
Fault Distance (km) ~ Building Height (m)         0.0580

Un |r| maximo de 0.08 sobre 1000 muestras indica ausencia total de relacion.


### 7.3 Correlacion de cada entrada con el indice de dano

In [11]:
print(df[INPUTS + [DAMAGE]].corr()[DAMAGE].sort_values(ascending=False).round(4).to_string())

Predicted Damage Index (0–1 Scale)      1.0000
Axial Stiffness (kN/m)                  0.0655
PGD (m)                                 0.0277
Fault Distance (km)                     0.0260
Spectral Acceleration (g)               0.0184
Number of Stories                       0.0151
PGV (m/s)                               0.0136
Damping Ratio (%)                       0.0122
PGA (m/s²)                              0.0062
Bending Stiffness (kN·m²)               0.0041
Building Height (m)                    -0.0049
Site Amplification Factor              -0.0087
Historical Earthquake Magnitude (Mw)   -0.0178
Mass of Structure (kg)                 -0.0191
Natural Frequency (Hz)                 -0.0309
Seismic Wave Frequency (Hz)            -0.0424


## 8. Verificacion de coherencia fisica frente a la NTE E.030

Se contrastan las relaciones que el documento de análisis exploratorio supone entre las variables.

### 8.1 Zona sismica frente a PGA

El Factor de Zona (Z) debería ordenar la aceleración del suelo: Z1 < Z2 < Z3 < Z4.

In [12]:
PGA = next(c for c in df.columns if c.startswith("PGA"))
print(df.groupby("Seismic Zone")[PGA].agg(["count", "mean", "min", "max"]).round(4).to_string())
print("\nLas medias son indistinguibles: la zona sismica no informa la aceleracion del suelo.")

              count   mean    min    max
Seismic Zone                            
High            232 0.8117 0.1019 1.4961
Low             258 0.7737 0.1000 1.4939
Moderate        230 0.8024 0.1117 1.4967
Very High       280 0.7897 0.1009 1.4994

Las medias son indistinguibles: la zona sismica no informa la aceleracion del suelo.


### 8.2 Tipo de suelo frente al factor de amplificacion de sitio

El perfil de suelo debería ordenar la amplificación (Roca < Arena < Arcilla < Suelo blando).

In [13]:
print(df.groupby("Soil Type")["Site Amplification Factor"].agg(["count", "mean"]).round(4).to_string())
print("\nLos perfiles de suelo tampoco ordenan la amplificacion.")

           count   mean
Soil Type              
Clay         259 1.8821
Rock         267 1.8857
Sand         263 1.8580
Soft Soil    211 1.8885

Los perfiles de suelo tampoco ordenan la amplificacion.


### 8.3 Numero de pisos frente a altura de la edificacion

Una altura típica de entrepiso de 3 m implicaría un crecimiento lineal de la altura con el número de niveles.

In [14]:
altura_piso = df["Building Height (m)"] / df["Number of Stories"]
print(f"Altura por piso: min = {altura_piso.min():.3f} m | max = {altura_piso.max():.2f} m | media = {altura_piso.mean():.2f} m")
print()
print(df.groupby(pd.cut(df["Number of Stories"], [0, 10, 20, 30, 40, 50]))["Building Height (m)"]
      .agg(["count", "mean"]).round(3).to_string())
print("\nLa altura no crece con el numero de pisos.")

Altura por piso: min = 0.161 m | max = 197.33 m | media = 10.34 m

                   count     mean
Number of Stories                
(0, 10]              233 102.0010
(10, 20]             185 102.6120
(20, 30]             218 108.4700
(30, 40]             197  99.7220
(40, 50]             167 106.6600

La altura no crece con el numero de pisos.


### 8.4 Deriva de entrepiso frente a los limites de la NTE E.030

Límites admisibles de distorsión: Albaililería 0.5 %, Concreto y Compuesto 0.7 %, Acero 1.0 %.

In [15]:
DRIFT = next(c for c in df.columns if c.startswith("Predicted Max Inter-Story Drift"))
MATERIAL = "Structural Material"
LIMITE = {"Masonry": 0.5, "Concrete": 0.7, "Composite": 0.7, "Steel": 1.0}

limite = df[MATERIAL].map(LIMITE)
excede = df[DRIFT] > limite
print(f"Deriva de entrepiso: media = {df[DRIFT].mean():.3f} % | max = {df[DRIFT].max():.3f} %")
print(f"Edificaciones que exceden el limite normativo: {int(excede.sum())} de {len(df)} ({excede.mean() * 100:.1f} %)\n")

resumen = (df.assign(excede=excede, limite=limite)
           .groupby(MATERIAL)
           .agg(n=(DRIFT, "size"),
                limite_normativo_pct=("limite", "first"),
                deriva_media_pct=(DRIFT, "mean"),
                deriva_max_pct=(DRIFT, "max"),
                excede_limite_pct=("excede", lambda s: round(s.mean() * 100, 1)))
           .round(3))
print(resumen.to_string())

Deriva de entrepiso: media = 2.514 % | max = 4.994 %
Edificaciones que exceden el limite normativo: 867 de 1000 (86.7 %)

                       n  limite_normativo_pct  deriva_media_pct  deriva_max_pct  excede_limite_pct
Structural Material                                                                                
Composite            241                0.7000            2.4960          4.9730            87.6000
Concrete             261                0.7000            2.5020          4.9840            88.1000
Masonry              240                0.5000            2.5410          4.9920            93.8000
Steel                258                1.0000            2.5170          4.9940            77.9000


## 9. Hallazgo principal y metodologia de trabajo

La extracción y caracterización demuestran que el dataset es una **muestra aleatoria uniforme de 1000 casos** en la que:

1. Las 21 variables numéricas son uniformes e independientes entre sí (correlación máxima |r| = 0.08).
2. Ninguna variable de entrada explica el índice de daño (|r| <= 0.07 frente a las 21 entradas).
3. El índice de daño y la probabilidad de colapso son la misma variable escalada.
4. Las seis variables de respuesta son mutuamente independientes, sin física que las relacione.
5. Las relaciones físicas esperadas (zona-PGA, suelo-amplificación, pisos-altura) no se cumplen.

**Consecuencia metodológica:** un modelo de aprendizaje supervisado no puede aprender una relación
inexistente. El desempeño de cualquier regresor será R2 cercano a 0, con un MAE equivalente a la
desviación estándar de la variable objetivo.

**Estrategia adoptada:** este archivo se utiliza como *fuente de distribuciones de entradas* (rangos y
distribuciones realistas de amenaza, sitio y propiedades estructurales) y las variables de respuesta se
construyen con un modelo físico propio, calibrado con la dinámica estructural y los criterios de la
NTE E.030. Esto habilita un entrenamiento supervisado con señal aprendible y físicamente
significativo para el contexto peruano.

**Código reusable:** las variables `df`, `NUM`, `CAT`, `INPUTS`, `DAMAGE`, `DRIFT` y `PGA` quedan
disponibles en el entorno para las celdas siguientes del análisis.

In [16]:
from scripts.extract_seismic import main
df, checks, prof = main()


Shape: (1000, 26)
Checks: {'nulos': 0, 'duplicados': 0, 'num_vars': 21, 'cat_vars': 5, 'uniformidad': {'PGA (m/s²)': 0.864, 'PGV (m/s)': 0.874, 'PGD (m)': 0.864, 'Spectral Acceleration (g)': 0.858, 'Site Amplification Factor': 0.875, 'Historical Earthquake Magnitude (Mw)': 0.868, 'Fault Distance (km)': 0.895, 'Seismic Wave Frequency (Hz)': 0.866, 'Building Height (m)': 0.867, 'Number of Stories': 0.84, 'Natural Frequency (Hz)': 0.881, 'Damping Ratio (%)': 0.868, 'Mass of Structure (kg)': 0.862, 'Axial Stiffness (kN/m)': 0.877, 'Bending Stiffness (kN·m²)': 0.88, 'Predicted Max Inter-Story Drift Ratio (%)': 0.856, 'Predicted Max Roof Displacement (m)': 0.865, 'Predicted Base Shear Force (kN)': 0.863, 'Predicted Structural Acceleration (m/s²)': 0.854, 'Predicted Damage Index (0–1 Scale)': 0.874, 'Predicted Collapse Probability (%)': 0.874}}
Profile guardado en: C:\Users\antho\Desktop\GRUPO-06\data\seismic_profile.json
